In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# STEP 1: GENERATE SYNTHETIC DATABASE (FROM THE OS DATASET)
def generate_dummy_dark_store_data(num_rows=10000):
    np.random.seed(42)
    sku_ids = [f"ACS-{np.random.randint(10000, 99999)}" for _ in range(num_rows)]
    dark_stores = np.random.choice(["DS-BLR-01", "DS-BLR-02", "DS-BLR-03"], size=num_rows)
    temp_zones = np.random.choice(["Ambient", "Chilled", "Frozen"], size=num_rows, p=[0.6, 0.3, 0.1])
    
    on_hand = np.random.randint(5, 200, size=num_rows)
    reserved = np.array([np.random.randint(0, max(1, int(oh * 0.25))) for oh in on_hand])
    available = on_hand - reserved
    reorder_level = np.random.randint(10, 60, size=num_rows)
    
    daily_velocity = np.round(np.random.uniform(0.5, 30.0, size=num_rows), 2)
    days_cover = np.round(available / daily_velocity, 2)
    urgent_reorder = (available < reorder_level).astype(int)
    
    df = pd.DataFrame({
        "SKU_ID": sku_ids, "Dark_Store_ID": dark_stores, "Temp_Zone": temp_zones,
        "On_Hand": on_hand, "Reserved": reserved, "Available": available,
        "Reorder_Level": reorder_level, "Daily_Velocity": daily_velocity, 
        "Days_Cover": days_cover, "Urgent_Reorder": urgent_reorder   
    })
    return df

# Create data
dataset = generate_dummy_dark_store_data(10000)

# STEP 2: PREPROCESSING & DATA SPLITTING
processed_data = pd.get_dummies(dataset, columns=["Dark_Store_ID", "Temp_Zone"])
#X = processed_data.drop(columns=["SKU_ID", "Days_Cover", "Urgent_Reorder"])
X = processed_data.drop(columns=["SKU_ID", "Available", "Reorder_Level", "Days_Cover", "Urgent_Reorder"])
y = processed_data["Urgent_Reorder"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# STEP 3: INITIALIZE AND TRAIN THE VARIABLE ('xgb_model')
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    verbosity=1,
    random_state=42,
    eval_metric='logloss' 
)
xgb_model.fit(X_train, y_train)
print("Model training successfully completed!")

# STEP 4: SAVE THE MODEL ARTIFACT (This fixed your error!)
joblib.dump(xgb_model, 'predictive_replenishment_model.pkl')
print("Artifact successfully saved as 'predictive_replenishment_model.pkl'")

# STEP 5: PRODUCTION INFERENCE WRAPPER FUNCTIONS
def check_dark_store_inventory_health(live_store_features_json):
    # Load model artifact securely from internal storage
    model = joblib.load('predictive_replenishment_model.pkl')
    live_features_df = pd.DataFrame([json.loads(live_store_features_json)])

    expected_features = model.get_booster().feature_names
    live_features_df = live_features_df.reindex(columns=expected_features, fill_value=0)
    
    prediction = model.predict(live_features_df)[0]
    prediction_proba = model.predict_proba(live_features_df)[0][1]
    
    if prediction == 1:
        reorder_alert_event = {
            "event_type": "AUTOMATED_PO_TRIGGERED",
            "confidence_score": round(float(prediction_proba), 4),
            "action": "GENERATE_PURCHASE_ORDER"
        }
        return json.dumps(reorder_alert_event)
    return json.dumps({"status": "Inventory levels stable."})

Model training successfully completed!
Artifact successfully saved as 'predictive_replenishment_model.pkl'


In [9]:
# STEP 6: VERIFY AND RUN LIVE INFERENCE ON DUMMY EVENTS

# Scenario 1: A low-stock event based on Tata Tea Premium (ACS-45566) at DS-BLR-01
# Physical Context: Available stock is 1, which is way below the Reorder Level of 50.
low_stock_event_json = json.dumps({
    "On_Hand": 1,
    "Reserved": 0,
    #"Available": 1,
    #"Reorder_Level": 50,
    "Daily_Velocity": 10.00, # Simulated feature addition
    "Dark_Store_ID_DS-BLR-01": 1, "Dark_Store_ID_DS-BLR-02": 0, "Dark_Store_ID_DS-BLR-03": 0,
    "Temp_Zone_Ambient": 1, "Temp_Zone_Chilled": 0, "Temp_Zone_Frozen": 0
})

# Scenario 2: A perfectly healthy stock event based on Whisper Sanitary Pads (ACS-99682)
# Physical Context: Available stock is 106, which is safe above the Reorder Level of 58.
healthy_stock_event_json = json.dumps({
    "On_Hand": 117,
    "Reserved": 11,
    #"Available": 106,
    #"Reorder_Level": 58,
    "Daily_Velocity": 14.32, # Simulated feature addition
    "Dark_Store_ID_DS-BLR-01": 1, "Dark_Store_ID_DS-BLR-02": 0, "Dark_Store_ID_DS-BLR-03": 0,
    "Temp_Zone_Ambient": 1, "Temp_Zone_Chilled": 0, "Temp_Zone_Frozen": 0
})

# Run live mock predictions through your function
print("--- Ingesting Low Stock Telemetry Payload ---")
low_stock_response = check_dark_store_inventory_health(low_stock_event_json)
print(low_stock_response, "\n")

print("--- Ingesting Healthy Stock Telemetry Payload ---")
healthy_stock_response = check_dark_store_inventory_health(healthy_stock_event_json)
print(healthy_stock_response)

--- Ingesting Low Stock Telemetry Payload ---
{"event_type": "AUTOMATED_PO_TRIGGERED", "confidence_score": 0.9928, "action": "GENERATE_PURCHASE_ORDER"} 

--- Ingesting Healthy Stock Telemetry Payload ---
{"status": "Inventory levels stable."}
